# Fitting SANS data

Previously, some small angle neutron scattering (SANS) data has been [simulated](./../3-mcstas/mcstas-sans.ipynb) and [reduced](./../4-reduction/reduction-sans.ipynb), and can now be analysed.
Before the analysis can begin, it is necessary to load the experimental data and check that it looks reasonable. 
The data can be loaded with `np.loadtxt` as the data has been stored in a simple space-separated column file. 

In [ ]:
import numpy as np
import utils
from sans_fitter import SANSFitter

In [ ]:
import warnings
import plotly.io as pio

# Ensuring figures are rendered as "png" makes them show up in the built book
# If we render them as interactive plots for notebooks (using "notebook"),
# it breaks the rendering of mathematical expressions in markdown cells.
pio.renderers.default = "png"

# Catch unwanted warnings
warnings.filterwarnings("ignore", category=UserWarning, module="bumps")
warnings.filterwarnings("ignore", category=UserWarning, message="Message serialization failed")

In [ ]:
filename = "../4-reduction/sans_iofq.dat"

⚠️ **If you did not complete the SANS data reduction yesterday**,
you can use some pre-prepared data by uncommenting and running the cell below:

In [ ]:
# filename = utils.fetch_data("4-reduction/sans_iofq.dat")

In [ ]:
q, i, di = utils.load(filename)

With the data read in, we can produce a quick plot simply using matplotlib. 

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.errorbar(q, i, di, fmt=".")
ax.set(yscale="log", xlabel="$q$/Å^-1", ylabel="I(q)")
plt.show()

We now want to consider the mathematical model to be used in the analysis. 
There are SANS models for myriad systems, see [the models in SasView](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/models/index.html). 
Initially, we will assume that our data has arisen from a spherical scattering object. 

The mathematical model for a sphere is:

$$
I(q) = \frac{\text{scale}}{V} \bigg(\frac{3 V \Delta \rho [\sin{(qr)} - qr \cos{(qr)}]}{(qr)^3}\bigg)^2 + \text{background}
$$ (sphere)

where $\text{scale}$ is a scale factor, $V$ is the volume of the sphere, $\Delta \rho$ is the difference between the solvent and particle scattering length density ($\rho_{solvent}$ - $\rho_{scatterer}$), $r$ is the radius of the sphere, a uniform background is added, and finally $q$ is the *q*-vector that the intensity is being calculated for.

### Exercise 1: simplify the expression

The mathematical model described in Eqn. {eq}`sphere` has five parameters. 
What simple mathematical simplification can be performed to reduce this to four?

**Solution:**

```{toggle}
The volume of a sphere is related to the radius of the sphere as 

$$
V = \frac{4}{3} \pi r^3. 
$$ (volume-sphere)

Therefore, the parameter $V$ can be replaced with Eqn. {eq}`volume-sphere`.
```

### Exercise 2: write a function that computes for the form factor of a sphere

Four parameters is a suitable number for modelling. 
Therefore, we should write a function that implements your reduced dimensionality version of Eqn. {eq}`sphere`.

In [ ]:
def sphere(q):
    """
    The function for the form factor of a sphere.

    Parameters
    ----------
    q:
        q-vectors to calculate for.

    Returns
    -------
    :
        The modelled intensity.
    """
    qr = q * radius
    V = 4 / 3 * np.pi * radius**3
    return (
        scale
        / V
        * (3 * V * delta_rho * (np.sin(qr) - qr * np.cos(qr)) / ((qr) ** 3)) ** 2
        + background
    )

### Exercise 3: create fitting parameters

`sans-fitter` provides a notebook-based backend for interaction with `sasmodels`, the most comprehensive community library of fitting functions for small-angle scattering data (used in the popular `SasView` GUI application). 

All of the functions here are 100% equivalent to working in the `SasView` GUI : You could even complete these tasks using the GUI, if you are more comfortable. However, we encourage you to try the Jupyter notebook approach below for these exercises! 

We imported `sans-fitter` at the top of the notebook. Now, one can create the fitter, load the SANS dataset above, and then fetch the sphere model and its default parameters:

In [ ]:
fitter = SANSFitter()

fitter.load_data(filename)
fitter.set_model("sphere")
fitter.get_params()

Knowing the parameters, we can proceed to assign sensible initial values, uniform prior distributions, define whether the parameters are to be fitted or fixed, and then confirm that we have passed these correctly.

You will note that `sans-fitter` identifies the presence, or absence, of experimental intensity error (dI) and resolution (dQ) in the provided data file. Accounting for aspects such as these complicates the implementation of fitting, beyond just applying a simple sphere form factor, like we derived earlier. 

In [ ]:
fitter.set_param("sld", value=3, min=1, max=30, vary=False)
fitter.set_param("sld_solvent", value=6, min=1, max=30, vary=False)

fitter.set_param("radius", value=80, min=10, max=300, vary=True)
fitter.set_param("scale", value=1.4e-7, min=0, max=1, vary=True)
fitter.set_param("background", value=0.1, min=0, max=1, vary=True)

fitter.get_params()

### Exercise 4: fit the data with the sphere function

Using `sans-fitter`, we can now fit the data and obtain maximum likelihood estimates for the varied parameters of the model.

We can start by using the BUMPS engine with the Nelder-Mead simplex method. 

In [ ]:
result = fitter.fit(engine="bumps", method="amoeba")
fitter.plot_results(show_residuals=True, log_scale=True)

Does the fit look sensible and is the model appropriate? 

Are there any ambiguities– are the residuals fine? Could the model be improved?

### Exercise 5: fit the data to an ellipsoid model 

In the same way as you have now learned, set the model and find the fitting parameters. Set them to sensible initial values, decide on reasonable ranges, and whether they should be free for the algorithm to optimise, or fixed. 

**Solution:**

In [ ]:
fitter2 = SANSFitter()
fitter2.load_data(filename)

fitter2.set_model("ellipsoid")
fitter2.get_params()

fitter2.set_param("sld", value=3, min=1, max=30, vary=False)
fitter2.set_param("sld_solvent", value=6, min=1, max=30, vary=False)

fitter2.set_param("radius_polar", value=80, min=10, max=300, vary=True)
fitter2.set_param("radius_equatorial", value=80, min=10, max=300, vary=True)

fitter2.set_param("scale", value=1.4e-7, min=0, max=1, vary=True)
fitter2.set_param("background", value=0.1, min=0, max=1, vary=True)

result = fitter2.fit(engine="bumps", method="amoeba")
fitter2.plot_results(show_residuals=True, log_scale=True)

How do the models and their outputs compare? 

What is the most appropriate description of the data?

Once you are happy with these fits, continue to [Part II: priors and statistics](./9a-bayesian_sans.ipynb), where we revisit the same dataset using Bayesian methods to obtain full posterior distributions for the model parameters, rather than single best-fit values.
